# Session 06 practical lab: Regression challenge for expected market price

> Warning: This notebook is a teaching lab. The production baseline notebook remains `04_regression_audi_a3_germany.ipynb`.

Head of Data 101 uses one central idea: **the pipeline is the product**.

In this lab, regression creates an expected market price layer for used-vehicle acquisition analysis. The model output is `expected_price_eur`. The business signal is `expected_price_gap = actual_price_eur - expected_price_eur`.

A negative gap means the listing is cheaper than the model expects. A positive gap means the listing is more expensive than the model expects. The gap is not an automatic investment recommendation; it is a prioritization signal for analyst review.

Course narrative boundary:

- Regression predicts expected price.
- Classification remains separate and predicts the external `top_price` label or probability.
- Do not derive the classification target from regression outputs.
- BI later combines actual price, expected-price gap, and top-price outputs.


## How to work

This is not a coding course. Use AI-assisted coding when useful, but make sure you can explain every result.

For each challenge, read the business reason, write or adapt the code, inspect the output, and write a short business interpretation in English.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from google.cloud import bigquery

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.style.use("default")


In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "config" / "project_config.yaml").exists():
            return path
    return start


def load_project_config(config_path: Path) -> dict:
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / "config" / "project_config.yaml")

RANDOM_SEED = int(PROJECT_CONFIG.get("random_state", 42))
np.random.seed(RANDOM_SEED)

project_id = str(PROJECT_CONFIG.get("gcp_project_id", "albertheadofdata101")).strip()
dataset_id = str(PROJECT_CONFIG.get("bq_dataset", "autoscout_audi_a3_germany")).strip()
make_scope = str(PROJECT_CONFIG.get("make", "Audi")).strip()
model_scope = str(PROJECT_CONFIG.get("model", "A3")).strip()
country_scope = str(PROJECT_CONFIG.get("country", "Germany")).strip()

make_filter = make_scope.lower()
model_filter = model_scope.lower()
country_filter = country_scope.lower()

print("Project ID:", project_id)
print("Dataset ID:", dataset_id)
print("Configured scope:", make_scope, model_scope, country_scope)
print("Random seed:", RANDOM_SEED)


## Challenge 0 - Setup and data contract

**Suggested time:** 10 minutes

**Business reason:**  
Before modeling, a professional data scientist must know exactly which dataset contract is being consumed. Regression must consume the BigQuery analytical contract, not a local CSV or an improvised dataset.

**Objective:**  
Load the regression dataset from BigQuery, apply configured scope, validate available rows and columns, and define the modeling target.

**Expected tasks:**
- Load project configuration.
- Connect to BigQuery.
- Query `vw_regression_dataset`.
- Apply configured make/model/country scope.
- Fallback to make/model only if the configured country returns no rows.
- Rename `actual_price_eur` to `price_eur`.
- Inspect row count, columns, first rows, and missing values.

**Expected output format:**
- Row count.
- Scope used.
- List of available columns.
- First 5 rows.
- Missing values summary.

**Reflection questions:**
- Why do we consume the BigQuery view instead of a local CSV?
- What could break downstream if we silently changed the dataset contract here?
- Why do we rename `actual_price_eur` to `price_eur` only inside the notebook?


In [ ]:
client = bigquery.Client(project=project_id)


def run_query(sql: str, parameters=None) -> pd.DataFrame:
    job_config = None
    if parameters is not None:
        job_config = bigquery.QueryJobConfig(query_parameters=parameters)
    return client.query(sql, job_config=job_config).to_dataframe()


In [ ]:
# Challenge 0 scaffold.
# Ask your AI assistant to help you build this step, but make sure you can explain the result.

# Your code here:
# 1. Query INFORMATION_SCHEMA.COLUMNS for vw_regression_dataset.
# 2. Build a SELECT list with the required fields.
# 3. Include fuel_type only if it exists in the view.
# 4. Query the configured make/model/country scope.
# 5. If no rows are returned, query make/model only and print a clear warning.
# 6. Rename actual_price_eur to price_eur.
# 7. Display row count, scope used, columns, first rows, and missing values.

columns_sql = f"""
SELECT column_name
FROM `{project_id}.{dataset_id}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'vw_regression_dataset'
ORDER BY ordinal_position
"""

# available_columns_df = run_query(columns_sql)
# display(available_columns_df)

# df = ...
# scope_used = ...

# print("Rows loaded:", len(df))
# print("Scope used:", scope_used)
# print("Columns:", list(df.columns))
# display(df.head())
# display(df.isna().sum().to_frame("missing_values"))


## Challenge 1 - Visual market reading before modeling

**Suggested time:** 20 minutes

**Business reason:**  
A data scientist should read the market visually before fitting a model. Charts reveal relationships, outliers, non-linearity, and possible data issues that metrics alone can hide.

**Objective:**  
Inspect the relationship between price and the main explanatory variables.

**Expected tasks:**
- Plot `price_eur` vs `mileage_km`.
- Plot `price_eur` vs `age_years`.
- Plot `price_eur` vs `power_hp`.
- Identify visible relationships.
- Identify possible outliers or unusual market segments.
- Form one hypothesis about which feature will be most useful.

**Expected output format:**
- Three scatter plots.
- Three short visual insights.
- One hypothesis about the strongest predictor.

**Reflection questions:**
- Which chart shows the clearest relationship with price?
- Which chart shows possible non-linearity?
- Which points might be data errors, rare opportunities, or vehicles the current feature set cannot explain?


In [ ]:
# Challenge 1 scaffold.
# Create three scatter plots with clear titles, axis labels, and units.

# Your code here
# plot_df = df.sample(n=min(20000, len(df)), random_state=RANDOM_SEED)
#
# plt.figure(figsize=(8, 5))
# plt.scatter(plot_df["mileage_km"], plot_df["price_eur"], alpha=0.35)
# plt.title("Price vs mileage")
# plt.xlabel("Mileage (km)")
# plt.ylabel("Price (EUR)")
# plt.tight_layout()
# plt.show()
#
# Repeat for age_years and power_hp.


### Your visual insights

1. Mileage insight:
2. Age insight:
3. Power insight:
4. Hypothesis about the strongest predictor:


## Challenge 2 - Univariate regression baselines

**Suggested time:** 25 minutes

**Business reason:**  
Before building a complex model, a data scientist needs a simple baseline. A one-variable model is limited, but it helps isolate the signal of each feature.

**Objective:**  
Train simple one-feature regression models and compare their explanatory power.

**Expected tasks:**
Train and evaluate:
- `price_eur ~ mileage_km`
- `price_eur ~ age_years`
- `price_eur ~ power_hp`

Use train/test split with the random seed from configuration. Report R2, MAE, RMSE, intercept, coefficient, and a plain-English coefficient interpretation.

**Expected output format:**
A comparison table with one row per model and columns: `model_name`, `feature_set`, `r2_test`, `mae_test`, `rmse_test`, `intercept`, `coefficient`, `coefficient_interpretation`.

**Reflection questions:**
- Which single variable explains price best?
- Which model has the lowest euro error?
- Does the coefficient sign make business sense?
- Would any one-variable model be sufficient for decision support?


In [ ]:
# Challenge 2 scaffold.
# Ask your AI assistant to help you build the loop, but make sure you can explain each metric.

# Your code here
# target_column = "price_eur"
# univariate_features = ["mileage_km", "age_years", "power_hp"]
# train_df, test_df = train_test_split(df, test_size=0.20, random_state=RANDOM_SEED)
# comparison_rows = []
#
# for feature_name in univariate_features:
#     # Select X_train, X_test, y_train, y_test.
#     # Fit LinearRegression().
#     # Predict on the test set.
#     # Calculate R2, MAE, and RMSE.
#     # Save intercept, coefficient, and your English interpretation.
#     pass
#
# univariate_results_df = pd.DataFrame(comparison_rows)
# display(univariate_results_df)


### Your baseline conclusion

If you could use only one variable, which one would you choose and why? Is the error level acceptable for business decision support?


## Challenge 3 - Multivariate regression: first expected-price layer

**Suggested time:** 30 minutes

**Business reason:**  
A realistic expected-price baseline should control for several vehicle characteristics at the same time. Mileage, age, and power interact conceptually in the market even if the first model remains linear.

**Objective:**  
Build the first serious expected-price model.

**Expected tasks:**
Train `price_eur ~ mileage_km + age_years + power_hp` and compare against the univariate baselines. Report R2, MAE, RMSE, intercept, coefficients, actual vs predicted plot, residual distribution, and residuals vs predicted plot.

**Expected output format:**
- Metrics table including all previous models and the multivariate model.
- Coefficient interpretation in business language.
- One paragraph answering: Is this model good enough to act as an expected-price baseline?

**Reflection questions:**
- Did the multivariate model improve the euro error?
- Do the coefficient signs make economic sense?
- Does the model fail more for expensive vehicles, cheap vehicles, old vehicles, or high-power vehicles?
- Is the model useful enough to support BI?


In [ ]:
# Challenge 3 scaffold.
# Build the first expected-price layer with the three core features.

# Your code here
# basic_features = ["mileage_km", "age_years", "power_hp"]
# X_train = train_df[basic_features]
# X_test = test_df[basic_features]
# y_train = train_df["price_eur"]
# y_test = test_df["price_eur"]
#
# model_basic = LinearRegression()
# model_basic.fit(X_train, y_train)
# y_pred_basic = model_basic.predict(X_test)
#
# Calculate metrics, build a coefficient table, and create diagnostic plots.


In [ ]:
# Diagnostic plot scaffold.

# Your code here
# residuals = y_test - y_pred_basic
# Build actual vs predicted, residual histogram, and residuals vs predicted plots.


### Your multivariate conclusion

Is this model good enough to act as an expected-price baseline? What would you tell the business team about its strengths and limitations?


## Challenge 4 - Feature composition: helping the model understand market structure

**Suggested time:** 35 minutes

**Business reason:**  
The used-car market is rarely perfectly linear. Depreciation may be steeper for newer cars, mileage may have diminishing effects, and derived variables may better represent usage intensity.

**Objective:**  
Improve the linear model using controlled feature engineering before jumping to black-box models.

**Expected tasks:**
Create `log_mileage`, `age_squared`, `mileage_per_year`, `power_per_year`, and `interaction_age_mileage`. Train at least a basic model, a transformed model with `log_mileage` and `age_squared`, and optionally a composed model with usage and interaction terms.

**Expected output format:**
- Metrics table comparing basic and enhanced models.
- Explanation of which engineered features helped.
- One business interpretation of the improvement.
- One warning about overfitting or loss of interpretability.

**Reflection questions:**
- Did the transformed model reduce MAE?
- Which composed feature has the clearest business meaning?
- Which composed feature is harder to explain?
- Is the improvement large enough to justify extra complexity?


In [ ]:
# Challenge 4 scaffold.
# Create composed features in a copy of the data.

# Your code here
# model_data = df.copy()
# safe_age = model_data["age_years"].clip(lower=1)
# model_data["log_mileage"] = np.log1p(model_data["mileage_km"])
# model_data["age_squared"] = model_data["age_years"] ** 2
# model_data["mileage_per_year"] = model_data["mileage_km"] / safe_age
# model_data["power_per_year"] = model_data["power_hp"] / safe_age
# model_data["interaction_age_mileage"] = model_data["age_years"] * model_data["mileage_km"]
#
# Evaluate basic and enhanced linear models on the same train/test split.


### Your feature engineering conclusion

Which engineered feature has the clearest business meaning? Which feature is hardest to explain? Did the improvement justify the added complexity?


## Challenge 5 - Advanced regression models: do flexible models improve decision quality?

**Suggested time:** 35 minutes

**Business reason:**  
Flexible models can capture non-linear relationships and feature interactions automatically. However, they may reduce transparency. A professional data scientist must decide whether the accuracy gain justifies the interpretability cost.

**Objective:**  
Compare interpretable linear models with more flexible regression models.

**Expected tasks:**
Train and evaluate `DecisionTreeRegressor`, `RandomForestRegressor`, and `HistGradientBoostingRegressor` on the same train/test split. Use the same feature set as the best composed model where possible.

**Expected output format:**
- Model comparison table.
- Feature importance chart for Random Forest if available.
- Decision on whether the advanced model is worth the complexity.
- One paragraph explaining the trade-off between accuracy and interpretability.

**Reflection questions:**
- Which model has the lowest MAE?
- Is the improvement large enough to matter in euros?
- Which model would be easier to explain to a committee?
- Which model would you trust more for a BI decision layer?
- What could go wrong if we only chase lower error?


In [ ]:
# Challenge 5 scaffold.

# Your code here
# advanced_features = composed_features  # Or your selected best enhanced feature set.
# advanced_models = {
#     "Decision tree regressor": DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED),
#     "Random forest regressor": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1),
#     "Histogram gradient boosting regressor": HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_SEED),
# }
# Fit, predict, calculate metrics, and compare against the linear models.


In [ ]:
# Feature importance scaffold for Random Forest.

# Your code here
# if hasattr(random_forest_model, "feature_importances_"):
#     importance_df = pd.DataFrame({"feature": advanced_features, "importance": random_forest_model.feature_importances_})
#     importance_df = importance_df.sort_values("importance", ascending=True)
#     plt.figure(figsize=(8, 5))
#     plt.barh(importance_df["feature"], importance_df["importance"])
#     plt.title("Random Forest feature importance")
#     plt.xlabel("Importance")
#     plt.ylabel("Feature")
#     plt.tight_layout()
#     plt.show()


### Your advanced model conclusion

Which model would you choose for the expected-price layer? Is it more accurate, or just more complex? Would you trust it for a decision queue?


## Challenge 6 - From regression model to business decision queue

**Suggested time:** 25 minutes

**Business reason:**  
A regression model becomes useful when it produces a decision-support artifact. The expected-price layer should help analysts prioritize which listings deserve review.

**Objective:**  
Turn the selected model into an actionable expected-price layer and review queue.

**Expected tasks:**
Select the best model using MAE, R2, RMSE, residual behavior, interpretability, and usefulness for decision support. Train it on all available data. Generate `expected_price_eur`, `expected_price_gap`, and `expected_price_gap_pct`. Create top 10 cheap candidates, top 10 expensive candidates, and a review queue.

**Expected output format:**
- Final selected model name.
- Top 10 cheap candidates.
- Top 10 expensive candidates.
- Three candidate listings for analyst review.
- One model risk.
- One recommended next improvement.

**Reflection questions:**
- Which listings look most attractive relative to expected price?
- Which listings are probably just data/model anomalies?
- What information is missing before making a real acquisition decision?
- How should this output be shown in BI?


In [ ]:
# Challenge 6 scaffold.

# Your code here
# selected_model_name = "..."
# selected_feature_columns = [...]
# selected_model = ...
# selected_model.fit(model_data[selected_feature_columns], model_data["price_eur"])
# expected_price = selected_model.predict(model_data[selected_feature_columns])
#
# review_queue = model_data[["listing_id", "price_eur", "mileage_km", "age_years", "power_hp"]].copy()
# review_queue["expected_price_eur"] = expected_price
# review_queue["expected_price_gap"] = review_queue["price_eur"] - review_queue["expected_price_eur"]
# review_queue["expected_price_gap_pct"] = review_queue["expected_price_gap"] / review_queue["expected_price_eur"]
#
# Sort by expected_price_gap to find cheaper-than-expected and more-expensive-than-expected listings.


In [ ]:
# Optional teacher/demo table reminder.
# Do not overwrite the production table fact_expected_price_predictions from the baseline notebook.
# If your instructor asks you to write a demo table, use a teacher/demo-specific table name.

# teacher_demo_table = f"{project_id}.{dataset_id}.fact_expected_price_predictions_teacher_demo"


### Your decision queue conclusion

1. Final selected model:
2. Three candidate listings for analyst review:
3. One model risk:
4. One recommended next improvement:
5. What would you show in BI?


## Final discussion: what did we learn about regression as a decision-support layer?

Use these prompts for the final class discussion:

- Why did we inspect the market visually before modeling?
- Why did we build baselines before complex models?
- What is the business meaning of MAE, RMSE, and R2?
- How do coefficients help explain the model?
- How do residuals become possible mispricing signals?
- When does feature engineering help, and when does it make the model fragile?
- When is an advanced model worth the interpretability cost?
- How does `expected_price_gap` become a BI-ready decision layer?
- Why does model output still require analyst judgment?

Final message:

**A useful regression model does not end in a metric. It ends in a reference layer that helps people explain deviations, prioritize review, and make better decisions.**
